# CCS_RL · End-to-End Iterative Action-Q Training and Cinematic Replay

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZZZPhaethon/CCS_RL/blob/main/examples/colab_vessel_trajectory_demo.ipynb?v=cinematic-v2-20260804)

**From physics-constrained simulation and counterfactual data generation to model training, validation, and an interactive operational replay.**

This notebook follows the repository's main research problem: vessel dispatch in a three-vessel Northern Lights-style CCS transport and storage chain. The learned controller changes **vessel dispatch only**. Mass balance, capacities, sailing, loading, unloading, injection, and pressure constraints are always enforced by the same physical simulator.

The final section replays the trained controller with the same cinematic visualization pipeline used by [`physical_layer_dashboardV2.html`](https://zzzphaethon.github.io/CCS_RL/physical_layer_dashboardV2.html): an interactive Leaflet map, synchronized time slider, live component cards, vessel status, inventory chart, playback speed, and light/dark themes.

> The default run is a deliberately small `G0 → P1` experiment that can finish in Colab. It uses the real environment, action masks, economic accounting, training code, and V2 renderer, but a much smaller simulator budget than the paper experiments. Its numerical results are educational and must not be reported as the repository's formal E1 results.


## 0. What are we training?

Iterative Action-Q is neither a weather predictor nor behavior cloning of Greedy trajectories. It learns:

\[
Q(s,a)=10^{-5}\left(C_{\text{baseline}}(s)-C_{a}(s)\right)
\]

Here, `a` is one legal joint action for the three vessels. The baseline and candidate start from the **same decision state**. The candidate replaces the current action, then both trajectories are rolled to the episode boundary and evaluated with the same terminal-cleanup cost.

```text
scenario seed
     │
     ▼
Greedy roll-in ──► root state s
                     ├── FOLLOW / Greedy ──► episode end ──► baseline cost
                     ├── candidate a₁ ─────► episode end ──► candidate cost
                     ├── candidate a₂ ─────► episode end ──► candidate cost
                     └── ...
                              │
                              ▼
                 same-state action-value dataset
                              │
                              ▼
             ensemble distributional Action-Q model
                              │
                 confidence + margin safety gate
                              ▼
             held-out evaluation + cinematic replay
```

The notebook covers:

1. Installation and reproducibility settings.
2. The CCS environment, state vector, joint-action vocabulary, and legal-action masks.
3. Leakage-safe train and validation counterfactual datasets.
4. `.npz` loading, schema inspection, label auditing, and same-root grouping.
5. Five-head quantile Action-Q training with validation-based early stopping.
6. Paired evaluation against Greedy on unseen scenario seeds.
7. A new V2-style interactive replay generated from the trained controller.
8. An optional `P1 → G1 → P2` iteration.


## 1. Install the project

Colab clones the current `main` branch and refreshes an existing `/content/CCS_RL` checkout before installing the RL extras. This prevents a reused runtime from silently loading the legacy Phase 1 dashboard. A local notebook reuses the current checkout. The first installation can take a few minutes.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ZZZPhaethon/CCS_RL.git"

if IN_COLAB:
    REPO_ROOT = Path("/content/CCS_RL")
    if not (REPO_ROOT / ".git").exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1", "--branch", "main",
                REPO_URL, str(REPO_ROOT),
            ],
            check=True,
        )
    else:
        # A Colab VM can survive notebook reloads. Pin this run to the latest
        # origin/main so an older visualization module cannot be reused.
        subprocess.run(
            [
                "git", "-C", str(REPO_ROOT), "fetch", "--depth", "1",
                "origin", "main",
            ],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(REPO_ROOT), "checkout", "--detach", "FETCH_HEAD"],
            check=True,
        )
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(
        (
            path
            for path in candidates
            if (path / "pyproject.toml").is_file()
            and (path / "src" / "sim").is_dir()
        ),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError("Start the notebook from inside the CCS_RL repository.")

os.chdir(REPO_ROOT)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        ".[rl]",
        "pandas",
        "matplotlib",
        "seaborn",
        "tqdm",
    ],
    check=True,
)

# Make the editable checkout importable immediately in the current kernel.
for path in (REPO_ROOT, REPO_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

REPO_REVISION = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print(f"Repository : {REPO_ROOT}")
print(f"Revision   : {REPO_REVISION} (latest main in Colab)")
print(f"Python     : {sys.version.split()[0]}")
print(f"Runtime    : {'Google Colab' if IN_COLAB else 'local'}")


## 2. Reproducible configuration and compute budget

The default `QUICK_DEMO=True` configuration uses:

- a 168-hour episode;
- four training seeds, two validation seeds, and three independent evaluation seeds;
- three counterfactual roots per seed;
- every legal single-vessel override plus a small sample of two-vessel actions;
- at most 15 training epochs.

Setting `QUICK_DEMO=False` switches to a 720-hour episode and denser roots, but it is still smaller than the formal 3,200-root production budget. The final section summarizes the full experiment configuration.


In [ ]:
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

QUICK_DEMO = True
MODEL_SEED = 7
REWARD_SCALE = 1e-5
VARIANT = "future_mlp_mode_destination"
SCENARIO_PROTOCOL = "unified_window_v1"

if QUICK_DEMO:
    EPISODE_HOURS = 168
    TRAIN_SEEDS = [1500, 1501, 1502, 1503]
    VALIDATION_SEEDS = [3200, 3201]
    EVALUATION_SEEDS = [4100, 4101, 4102]
    ROOT_FRACTIONS = [0.25, 0.50, 0.75]
    MAX_TWO_VESSEL_ACTIONS = 2
    MAX_THREE_VESSEL_ACTIONS = 0
    MAX_EPOCHS = 15
    PATIENCE = 4
else:
    EPISODE_HOURS = 720
    TRAIN_SEEDS = list(range(1500, 1512))
    VALIDATION_SEEDS = list(range(3200, 3204))
    EVALUATION_SEEDS = list(range(4100, 4106))
    ROOT_FRACTIONS = [0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85]
    MAX_TWO_VESSEL_ACTIONS = 8
    MAX_THREE_VESSEL_ACTIONS = 4
    MAX_EPOCHS = 30
    PATIENCE = 6

random.seed(MODEL_SEED)
np.random.seed(MODEL_SEED)
torch.manual_seed(MODEL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MODEL_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_ROOT = (
    Path("/content/ccs_rl_iterative_q_demo")
    if IN_COLAB
    else REPO_ROOT / "output" / "colab_iterative_q_demo"
)
DATA_DIR = RUN_ROOT / "data"
MODEL_DIR = RUN_ROOT / "p1"
EVAL_DIR = RUN_ROOT / "evaluation"
for directory in (DATA_DIR,):
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_DATA = DATA_DIR / "g0_train.npz"
VALIDATION_DATA = DATA_DIR / "g0_validation.npz"
CHECKPOINT = MODEL_DIR / "iterative_action_q.pt"

config_view = {
    "quick_demo": QUICK_DEMO,
    "episode_hours": EPISODE_HOURS,
    "train_seeds": TRAIN_SEEDS,
    "validation_seeds": VALIDATION_SEEDS,
    "evaluation_seeds": EVALUATION_SEEDS,
    "root_fractions": ROOT_FRACTIONS,
    "device": DEVICE,
    "run_root": str(RUN_ROOT),
}
print(json.dumps(config_view, indent=2))


## 3. Build the real CCS environment and inspect its action space

`common.make_event_env` constructs:

- the `northern_lights_phase1_3vessels` physical network;
- the frozen `unified_window_v1` disturbance protocol;
- the economic objective;
- the shared automatic-max injection-well controller;
- Greedy as the residual baseline;
- the event-based joint residual-action wrapper.

Each vessel has native dispatch choices plus `FOLLOW`. Their Cartesian product defines the joint-action vocabulary. A state-dependent physical mask then removes illegal joint actions.


In [ ]:
from types import SimpleNamespace

from experiments import iterative_q_data_common as common

env_args = SimpleNamespace(
    episode_hours=EPISODE_HOURS,
    reward_scale=REWARD_SCALE,
    variant=VARIANT,
    scenario_protocol=SCENARIO_PROTOCOL,
    hard_scenario_probability=0.5,
    forecast_context_hours=168,
    future_summary_windows_h=[24, 72],
    stress_level="medium",
    seeds=TRAIN_SEEDS,
)

wrapper = common.make_event_env(env_args)
observation, reset_info = wrapper.reset_native_seed(TRAIN_SEEDS[0])
legal_mask = wrapper.action_masks()

environment_summary = {
    "emitters": wrapper.env.emitter_ids,
    "vessels": wrapper.env.vessel_ids,
    "terminals": wrapper.env.terminal_ids,
    "state_shape": tuple(observation["state"].shape),
    "state_features": len(common.state_feature_names(wrapper)),
    "joint_actions": int(wrapper.action_space.n),
    "legal_actions_at_first_event": int(legal_mask.sum()),
    "follow_action_index": int(wrapper.follow_action()),
    "episode_steps": int(wrapper.env.n_steps),
}
print(json.dumps(environment_summary, indent=2))

pd.DataFrame(
    {
        "feature_index": range(len(common.state_feature_names(wrapper))),
        "feature_name": common.state_feature_names(wrapper),
        "value_at_first_event": observation["state"],
    }
).head(20)


In [ ]:
# Preview legal joint actions. Each column is a vessel; values are local action indices.
joint_actions = wrapper._joint_action_array
legal_indices = np.flatnonzero(legal_mask)
action_preview = pd.DataFrame(
    joint_actions[legal_indices[:12]],
    columns=wrapper.env.vessel_ids,
    index=legal_indices[:12],
)
action_preview.index.name = "joint_action_index"
action_preview["is_FOLLOW"] = action_preview.index == wrapper.follow_action()
action_preview


## 4. Generate G0 counterfactual training data

For each scenario seed, the repository generator:

1. runs Greedy once and stores hourly rewards plus common economic metrics;
2. rolls Greedy into each root fraction;
3. keeps every legal one-vessel override and samples a few higher-order overrides;
4. deep-copies the same root and rolls every candidate to the episode boundary;
5. verifies that the residual return equals `baseline total cost − candidate total cost`;
6. writes a compressed `.npz` containing states, actions, masks, returns, costs, and complete metadata.

Training and validation use disjoint scenario seeds. This cell directly calls the production data generator in `experiments/generate_iterative_q_greedy_data.py`.


In [ ]:
from experiments import generate_iterative_q_greedy_data as g0_generator

def generate_g0(path: Path, split: str, seeds: list[int]):
    if path.exists():
        print(f"Reuse existing {split} dataset: {path}")
        return

    argv = [
        "--out-path", str(path),
        "--split", split,
        "--seeds", *map(str, seeds),
        "--root-fractions", *map(str, ROOT_FRACTIONS),
        "--roots-per-seed", str(len(ROOT_FRACTIONS)),
        "--max-two-vessel-actions", str(MAX_TWO_VESSEL_ACTIONS),
        "--max-three-vessel-actions", str(MAX_THREE_VESSEL_ACTIONS),
        "--episode-hours", str(EPISODE_HOURS),
        "--reward-scale", str(REWARD_SCALE),
        "--dataset-seed", str(20260723 + (split == "validation")),
        "--variant", VARIANT,
        "--device", "cpu",
        "--scenario-protocol", SCENARIO_PROTOCOL,
        "--hard-scenario-probability", "0.5",
        "--forecast-context-hours", "168",
        "--future-summary-windows-h", "24", "72",
        "--stress-level", "medium",
    ]
    args = g0_generator.parse_args(argv)
    started = time.perf_counter()
    summary = g0_generator.generate_dataset(args)
    print(f"{split} generation took {(time.perf_counter() - started):.1f} s")
    return summary

generate_g0(TRAIN_DATA, "train", TRAIN_SEEDS)
generate_g0(VALIDATION_DATA, "validation", VALIDATION_SEEDS)


## 5. Data loading: inspect the `.npz` arrays and metadata

This is not an ordinary one-step replay buffer. The first dimension contains candidate records. Records sharing `(scenario_seed, root_time_h)` belong to one same-state action group and are grouped before training.


In [ ]:
def load_npz(path: Path):
    with np.load(path, allow_pickle=False) as loaded:
        arrays = {key: loaded[key].copy() for key in loaded.files}
    metadata = json.loads(str(arrays.pop("metadata_json")))
    return arrays, metadata

train_raw, train_meta = load_npz(TRAIN_DATA)
validation_raw, validation_meta = load_npz(VALIDATION_DATA)

schema_rows = []
for name, value in train_raw.items():
    schema_rows.append(
        {
            "field": name,
            "shape": str(value.shape),
            "dtype": str(value.dtype),
            "example": (
                str(value.reshape(-1)[0])
                if value.size and value.dtype.kind in "biuf"
                else "—"
            ),
        }
    )
pd.DataFrame(schema_rows).sort_values("field").reset_index(drop=True)


In [ ]:
metadata_view = {
    key: train_meta[key]
    for key in [
        "kind",
        "split",
        "scenario_seeds",
        "episode_hours",
        "observation_variant",
        "reward_scale",
        "objective",
        "residual_reward",
        "uses_mpc",
        "scenario_protocol",
        "root_fractions",
        "roots_per_seed",
        "training_simulator_usage",
    ]
}
print(json.dumps(metadata_view, indent=2))


## 6. Audit split integrity, same-root states, and economic labels

The following checks are essential:

- training and validation scenario seeds do not overlap;
- every candidate in a root has exactly the same state;
- the scaled learning target agrees with the full economic cost difference.

Randomly splitting candidate rows would be invalid because different actions from the same root could leak into both training and validation.


In [ ]:
train_seed_set = set(map(int, train_meta["scenario_seeds"]))
validation_seed_set = set(map(int, validation_meta["scenario_seeds"]))
assert train_seed_set.isdisjoint(validation_seed_set)

def audit_dataset(data, metadata, name):
    keys = np.column_stack([data["scenario_seed"], data["root_time_h"]])
    unique_roots = np.unique(keys, axis=0)

    same_root_ok = True
    for seed, root_h in unique_roots:
        idx = np.flatnonzero(
            (data["scenario_seed"] == seed)
            & (data["root_time_h"] == root_h)
        )
        same_root_ok &= np.allclose(
            data["states"][idx, 0],
            data["states"][idx[0], 0],
        )

    target_eur = data["return_to_go"][:, 0] / metadata["reward_scale"]
    exact_saving_eur = (
        data["baseline_total_cost_eur"]
        - data["candidate_total_cost_eur"]
    )
    max_label_error_eur = float(
        np.max(np.abs(target_eur - exact_saving_eur))
    )
    return {
        "split": name,
        "scenario_seeds": len(set(map(int, data["scenario_seed"]))),
        "candidate_records": len(data["actions"]),
        "same_state_roots": len(unique_roots),
        "same_root_state_check": bool(same_root_ok),
        "max_label_alignment_error_eur": max_label_error_eur,
        "improving_candidates": int((exact_saving_eur > 1e-6).sum()),
        "worse_candidates": int((exact_saving_eur < -1e-6).sum()),
    }

audit_table = pd.DataFrame(
    [
        audit_dataset(train_raw, train_meta, "train"),
        audit_dataset(validation_raw, validation_meta, "validation"),
    ]
)
audit_table


In [ ]:
# Full-episode candidate savings; positive values are better than Greedy.
train_saving_eur = (
    train_raw["baseline_total_cost_eur"]
    - train_raw["candidate_total_cost_eur"]
)
root_counts = (
    pd.DataFrame(
        {
            "seed": train_raw["scenario_seed"],
            "root_h": train_raw["root_time_h"],
        }
    )
    .value_counts()
    .rename("candidate_actions")
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train_saving_eur / 1e3, bins=30, ax=axes[0], color="#0A7E8C")
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set(
    title="G0 candidate value distribution",
    xlabel="Full-episode saving vs Greedy (kEUR)",
)
sns.barplot(
    data=root_counts,
    x="root_h",
    y="candidate_actions",
    color="#D89B3C",
    ax=axes[1],
)
axes[1].set(
    title="Candidate actions per same-state root",
    xlabel="Physical root time (h)",
)
plt.tight_layout()
plt.show()


## 7. Convert candidate records into supervised action groups

`GroupedDenseActionDataset`:

- groups records by `(scenario_seed, root_time_h)`;
- adds `FOLLOW` with a target of zero;
- pads roots with different numbers of legal candidates;
- returns `(state, actions, targets, valid_mask)`.

The network therefore learns full-episode action values and within-root action rankings, not a one-hour reward.


In [ ]:
from scripts.train_iterative_action_q import GroupedDenseActionDataset

grouped_train = GroupedDenseActionDataset(
    train_raw,
    follow_action_index=int(train_meta["follow_action_index"]),
    observation_input="state_only",
    reward_scale=float(train_meta["reward_scale"]),
)
grouped_validation = GroupedDenseActionDataset(
    validation_raw,
    follow_action_index=int(validation_meta["follow_action_index"]),
    observation_input="state_only",
    reward_scale=float(validation_meta["reward_scale"]),
)

state, future, actions, targets, valid, root_h, anchor = grouped_train[0]
sample_root = pd.DataFrame(
    {
        "joint_action_index": actions[valid],
        "target_reward_units": targets[valid],
        "target_saving_eur": targets[valid] / REWARD_SCALE,
    }
).sort_values("target_saving_eur", ascending=False)

print(
    f"train roots={len(grouped_train)}, validation roots={len(grouped_validation)}, "
    f"state_dim={state.shape[0]}, root_h={root_h}"
)
sample_root.head(12)


## 8. Train P1: an ensemble distributional Action-Q model

The model contains:

- an entity-aware state encoder;
- structured per-vessel action embeddings;
- five bootstrap heads;
- 21 quantiles per action in quick mode (51 in the formal configuration);
- a frozen randomized prior;
- quantile regression, pairwise ranking, listwise ranking, sign classification, and a FOLLOW anchor.

Checkpoint selection uses the validation composite:

`balanced_sign_accuracy + pairwise_accuracy + top1_improving_fraction + 0.05 × R²`

The cell reuses an existing checkpoint if present. Change `RUN_ROOT` or remove that run directory to retrain from scratch.


In [ ]:
from scripts import train_iterative_action_q as trainer

if CHECKPOINT.exists():
    print(f"Reuse existing checkpoint: {CHECKPOINT}")
    training_summary = json.loads(
        (MODEL_DIR / "summary.json").read_text(encoding="utf-8")
    )
else:
    train_argv = [
        "--train-data", str(TRAIN_DATA),
        "--validation-data", str(VALIDATION_DATA),
        "--out-dir", str(MODEL_DIR),
        "--observation-input", "state_only",
        "--epochs", str(MAX_EPOCHS),
        "--patience", str(PATIENCE),
        "--batch-size", "4",
        "--heads", "5",
        "--quantiles", "21" if QUICK_DEMO else "51",
        "--encoder-learning-rate", "0.0003",
        "--head-learning-rate", "0.0005",
        "--model-seed", str(MODEL_SEED),
        "--device", DEVICE,
    ]
    train_args = trainer.parse_args(train_argv)
    started = time.perf_counter()
    training_summary = trainer.run(train_args)
    print(f"Training took {(time.perf_counter() - started):.1f} s")

print(f"Checkpoint: {training_summary['checkpoint']}")


In [ ]:
# Plot the validation metrics used by checkpoint selection and early stopping.
history_rows = []
for row in training_summary["history"]:
    history_rows.append(
        {
            "epoch": row["epoch"],
            "train_loss": row["train_loss"],
            "selection_score": row["selection_score"],
            "pairwise_accuracy": row["validation"]["pairwise_accuracy"],
            "balanced_sign_accuracy": row["validation"]["balanced_sign_accuracy"],
            "top1_improving_fraction": row["validation"]["top1_improving_fraction"],
            "mean_regret_eur": (
                row["validation"]["mean_regret"] / REWARD_SCALE
            ),
        }
    )
history_df = pd.DataFrame(history_rows)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.lineplot(data=history_df, x="epoch", y="train_loss", marker="o", ax=axes[0])
sns.lineplot(data=history_df, x="epoch", y="selection_score", marker="o", ax=axes[1])
history_df.plot(
    x="epoch",
    y=["pairwise_accuracy", "balanced_sign_accuracy", "top1_improving_fraction"],
    marker="o",
    ax=axes[2],
)
axes[0].set_title("Training objective")
axes[1].set_title("Validation selection score")
axes[2].set_title("Validation decision metrics")
axes[2].set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

history_df


## 9. Inspect the checkpoint and held-out-root validation

The trainer restores the **best validation epoch**, not the final epoch. The checkpoint stores weights, normalization statistics, action schema, data provenance, configuration, and validation metrics so that the model remains auditable.


In [ ]:
checkpoint_payload = torch.load(
    CHECKPOINT, map_location="cpu", weights_only=False
)
checkpoint_overview = {
    "top_level_keys": list(checkpoint_payload),
    "q_head": checkpoint_payload["configuration"]["q_head"],
    "observation_input": checkpoint_payload["configuration"]["observation_input"],
    "heads": checkpoint_payload["configuration"]["heads"],
    "quantiles": checkpoint_payload["configuration"]["quantiles"],
    "state_features": len(checkpoint_payload["metadata"]["state_feature_names"]),
    "joint_actions": len(checkpoint_payload["metadata"]["joint_actions"]),
    "return_scale": checkpoint_payload["normalization"]["return_scale"],
    "train_sources": checkpoint_payload["metadata"]["training_data_sources"],
    "validation_sources": checkpoint_payload["metadata"]["validation_data_sources"],
}
print(json.dumps(checkpoint_overview, indent=2))

final_validation = pd.Series(
    training_summary["final_validation"], name="held-out-root validation"
)
final_validation[
    [
        "mae",
        "rmse",
        "r2",
        "balanced_sign_accuracy",
        "pairwise_accuracy",
        "top1_improving_fraction",
        "top1_non_worse_fraction",
        "top1_mean_return",
        "oracle_mean_return",
        "mean_regret",
        "groups",
    ]
]


## 10. Independent scenario validation: gated policy versus Greedy

Dense-action validation asks whether the model ranks actions at unseen roots. Full-episode evaluation asks whether the gated controller improves actual economic outcomes on unseen scenarios.

Two gates are evaluated:

- `demo_loose`: one supporting head, zero margin, at most six overrides;
- `demo_safe`: at least four of five heads, a 0.10 margin (about €10k), at most four overrides.

Formal E1 uses a 720-hour protocol, a larger data budget, locked gates, multiple model seeds, and a separate evaluation block. This notebook does not access the formal-test seeds.


In [ ]:
from experiments import evaluate_iterative_action_q as policy_evaluator

if (EVAL_DIR / "summary.json").exists():
    print(f"Reuse existing evaluation: {EVAL_DIR}")
    policy_summary = json.loads(
        (EVAL_DIR / "summary.json").read_text(encoding="utf-8")
    )
else:
    eval_argv = [
        "--checkpoint", str(CHECKPOINT),
        "--out-dir", str(EVAL_DIR),
        "--eval-seeds", *map(str, EVALUATION_SEEDS),
        "--episode-hours", str(EPISODE_HOURS),
        "--reward-scale", str(REWARD_SCALE),
        "--gates",
        "demo_loose:1:0.0:6",
        "demo_safe:4:0.10:4",
        "--device", DEVICE,
        "--scenario-protocol", SCENARIO_PROTOCOL,
        "--hard-scenario-probability", "0.5",
        "--forecast-context-hours", "168",
        "--future-summary-windows-h", "24", "72",
        "--stress-level", "medium",
    ]
    eval_args = policy_evaluator.parse_args(eval_argv)
    started = time.perf_counter()
    policy_summary = policy_evaluator.run(eval_args)
    print(f"Policy evaluation took {(time.perf_counter() - started):.1f} s")

evaluation_df = pd.read_csv(EVAL_DIR / "evaluation.csv")
evaluation_df[
    [
        "gate",
        "seed",
        "total_cost_eur",
        "greedy_total_cost_eur",
        "delta_total_cost_eur",
        "vented_t",
        "greedy_vented_t",
        "override_events",
    ]
]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.barplot(
    data=evaluation_df,
    x="seed",
    y="delta_total_cost_eur",
    hue="gate",
    ax=axes[0],
)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set(
    title="Paired cost delta vs Greedy",
    ylabel="Controller − Greedy (EUR)",
)

sns.scatterplot(
    data=evaluation_df,
    x="greedy_vented_t",
    y="vented_t",
    hue="gate",
    s=90,
    ax=axes[1],
)
limit = max(
    1.0,
    evaluation_df[["greedy_vented_t", "vented_t"]].to_numpy().max(),
)
axes[1].plot([0, limit], [0, limit], "--", color="grey")
axes[1].set(title="Venting: controller vs Greedy")

sns.barplot(
    data=evaluation_df,
    x="seed",
    y="override_events",
    hue="gate",
    ax=axes[2],
)
axes[2].set(title="Accepted learned overrides")

plt.tight_layout()
plt.show()

summary_rows = []
for gate, values in policy_summary["summary"].items():
    summary_rows.append({"gate": gate, **values})
pd.DataFrame(summary_rows)[
    [
        "gate",
        "episodes",
        "mean_delta_total_cost_eur",
        "mean_delta_95pct_ci_eur",
        "wins",
        "ties",
        "losses",
        "mean_vented_t",
        "mean_override_events",
    ]
]


### How to interpret the evaluation

- `delta_total_cost_eur < 0` means the learned controller is cheaper than Greedy.
- A small-data model can be worse than Greedy. That is evidence that validation and safety gates are necessary, not that the pipeline failed.
- A three-seed bootstrap interval has no inferential value; it is shown only to exercise the evaluation code.
- Zero overrides from `demo_safe` is a valid result. Falling back to Greedy under uncertainty is an intentional design property.


## 11. Generate a V2 cinematic replay from the trained controller

This section uses the same plotting architecture as the project's new [`physical_layer_dashboardV2.html`](https://zzzphaethon.github.io/CCS_RL/physical_layer_dashboardV2.html):

```text
hourly physical states
    → TraceRecorder
    → replay_trace.csv
    → build_e1_cinematic_payload
    → render_cinematic_dashboard_html
    → standalone interactive HTML
```

The controller is replayed on the first unseen evaluation seed. Every native simulator hour is recorded, including hours automatically advanced between event decisions. The trace contains:

- emitter, vessel, and terminal inventories;
- vessel operational states and destinations;
- cumulative venting and weather speed factor;
- capture availability and injection-well status.

The resulting dashboard uses the V2 Leaflet map, component browser, synchronized timeline, vessel selector, cargo chart, playback controls, and light/dark theme. It visualizes this notebook's trained P1 checkpoint rather than embedding the repository's archived E1 replay.


In [ ]:
import csv

from experiments.plot_e1_figure4 import TraceRecorder
from sim.visualization import write_e1_cinematic_dashboard

REPLAY_SEED = EVALUATION_SEEDS[0]
REPLAY_DIR = RUN_ROOT / "cinematic_replay"
REPLAY_DIR.mkdir(parents=True, exist_ok=True)
REPLAY_TRACE = REPLAY_DIR / "colab_p1_hourly_trace.csv"
REPLAY_HTML = REPLAY_DIR / "colab_p1_cinematic_dashboard.html"

replay_argv = [
    "--checkpoint", str(CHECKPOINT),
    "--out-dir", str(RUN_ROOT / "unused_replay_evaluation"),
    "--eval-seeds", str(REPLAY_SEED),
    "--episode-hours", str(EPISODE_HOURS),
    "--reward-scale", str(REWARD_SCALE),
    "--gates", "cinematic_safe:4:0.10:4",
    "--device", DEVICE,
    "--scenario-protocol", SCENARIO_PROTOCOL,
    "--hard-scenario-probability", "0.5",
    "--forecast-context-hours", "168",
    "--future-summary-windows-h", "24", "72",
    "--stress-level", "medium",
]
replay_args = policy_evaluator.parse_args(replay_argv)
replay_device = torch.device(DEVICE)
replay_model, replay_metadata = policy_evaluator._load_model(
    replay_args, replay_device
)
replay_variant = str(replay_metadata["observation_variant"])
replay_follow = int(replay_metadata["follow_action_index"])
replay_gate = replay_args.gates[0]
replay_wrapper = policy_evaluator.make_event_env(
    replay_args, replay_variant
)

# Reset the residual environment before its event wrapper skips forced hours.
replay_observation, replay_info = (
    replay_wrapper.residual_env.reset_native_seed(REPLAY_SEED)
)
neutral_high_output = {
    emitter_id: [1.0] * (EPISODE_HOURS + 169)
    for emitter_id in replay_wrapper.env.emitter_ids
}
recorder = TraceRecorder(neutral_high_output)
recorder.record(replay_wrapper.env)

# Record every native hourly transition, including auto-advanced hours.
original_native_step = replay_wrapper.env.step

def recorded_native_step(action):
    result = original_native_step(action)
    recorder.record(replay_wrapper.env)
    return result

replay_wrapper.env.step = recorded_native_step
replay_observation, replay_info = replay_wrapper._after_reset(
    replay_observation, replay_info
)

done = False
replay_events = 0
replay_overrides = 0
while not done:
    expected_q = policy_evaluator.expected_q_for_observation(
        replay_model,
        replay_observation,
        replay_wrapper.env,
        replay_device,
        replay_args.future_ablation,
    )
    action, decision = policy_evaluator.select_safe_action(
        expected_q,
        replay_wrapper.action_masks(),
        replay_follow,
        required_heads=int(replay_gate["required_heads"]),
        margin=float(replay_gate["margin"]),
        uncertainty_beta=float(replay_gate["uncertainty_beta"]),
    )
    if (
        action != replay_follow
        and replay_overrides >= int(replay_gate["max_overrides"])
    ):
        action = replay_follow

    replay_observation, _, terminated, truncated, replay_info = (
        replay_wrapper.step(action)
    )
    replay_events += 1
    replay_overrides += int(action != replay_follow)
    done = bool(terminated or truncated)

expected_hours = [float(hour) for hour in range(EPISODE_HOURS + 1)]
recorded_hours = [float(frame["hour"]) for frame in recorder.frames]
assert recorded_hours == expected_hours, (
    f"Replay is not hourly-contiguous: {len(recorded_hours)} frames"
)

for frame in recorder.frames:
    frame["controller"] = "colab_iterative_action_q_p1"
    frame["test_seed"] = REPLAY_SEED

with REPLAY_TRACE.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(recorder.frames[0]))
    writer.writeheader()
    writer.writerows(recorder.frames)

write_e1_cinematic_dashboard(
    REPLAY_HTML,
    REPLAY_TRACE,
    title="Iterative Action-Q · Northern Lights operations",
)
print(
    json.dumps(
        {
            "replay_seed": REPLAY_SEED,
            "hourly_frames": len(recorder.frames),
            "event_decisions": replay_events,
            "accepted_overrides": replay_overrides,
            "trace_csv": str(REPLAY_TRACE),
            "dashboard_html": str(REPLAY_HTML),
            "dashboard_size_mb": round(REPLAY_HTML.stat().st_size / 1e6, 2),
        },
        indent=2,
    )
)


In [ ]:
import html as html_module

from IPython.display import HTML, display

# srcdoc keeps the standalone V2 HTML self-contained inside Colab.
dashboard_document = REPLAY_HTML.read_text(encoding="utf-8")
required_v2_markers = (
    'id="cinematicMap"',
    'id="playPause"',
    'class="theme-switcher"',
    "CCS fleet operations",
)
missing_v2_markers = [
    marker for marker in required_v2_markers if marker not in dashboard_document
]
if missing_v2_markers or "Northern Lights Phase 1 Dashboard" in dashboard_document:
    raise RuntimeError(
        "The generated file is not the cinematic V2 dashboard. "
        f"Missing markers: {missing_v2_markers}. Restart the runtime and Run all."
    )

print("Verified: cinematic V2 renderer (not the legacy Phase 1 dashboard).")
display(
    HTML(
        f'''
        <iframe
          srcdoc="{html_module.escape(dashboard_document, quote=True)}"
          style="width:100%; height:820px; border:1px solid #d8e1df;
                 border-radius:14px; background:#f7faf9;"
          loading="lazy"
          allowfullscreen>
        </iframe>
        '''
    )
)


## 12. Optional: continue from P1 to G1 and P2

`G0 → P1` trains only on states visited by Greedy. Full Iterative Action-Q locks P1, collects G1 on states visited by P1 itself, and then cumulatively trains P2 on `G0 + G1`.

The cell is disabled by default because policy-roll-in counterfactual generation is more expensive. Set `RUN_P2=True` to:

1. create a SHA256 policy lock for P1;
2. generate anchored G1 train and validation datasets on new seeds;
3. initialize P2 from P1 and train on cumulative G0+G1 data.

Iteration matters because it moves the training distribution toward the current policy's state-visitation distribution.


In [ ]:
RUN_P2 = False

if RUN_P2:
    import hashlib

    from experiments import generate_iterative_q_policy_data as policy_data

    G1_TRAIN = DATA_DIR / "g1_train.npz"
    G1_VALIDATION = DATA_DIR / "g1_validation.npz"
    LOCK_PATH = RUN_ROOT / "p1_lock.json"
    P2_DIR = RUN_ROOT / "p2"

    # Demo policy windows; the formal 720-hour configuration uses eight windows.
    demo_windows = [
        [int(0.20 * EPISODE_HOURS), int(0.39 * EPISODE_HOURS)],
        [int(0.40 * EPISODE_HOURS), int(0.59 * EPISODE_HOURS)],
        [int(0.60 * EPISODE_HOURS), int(0.85 * EPISODE_HOURS)],
    ]
    digest = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
    lock_payload = {
        "protocol_id": "colab_iterative_q_p1",
        "locked_checkpoint": str(CHECKPOINT.resolve()),
        "checkpoint_sha256": digest,
        "observation_input": "state_only",
        "policy": {
            "required_heads": 4,
            "residual_margin": 0.10,
            "economic_margin_eur": 10_000,
            "max_overrides": len(demo_windows),
            "one_override_per_window": True,
            "windows_h": demo_windows,
        },
        "uses_mpc_for_training_or_selection": False,
    }
    if not LOCK_PATH.exists():
        LOCK_PATH.write_text(
            json.dumps(lock_payload, indent=2) + "\n", encoding="utf-8"
        )

    def generate_g1(path, split, seeds):
        if path.exists():
            return
        argv = [
            "--lock-config", str(LOCK_PATH),
            "--out-path", str(path),
            "--split", split,
            "--seeds", *map(str, seeds),
            "--max-two-vessel-actions", "2",
            "--max-three-vessel-actions", "0",
            "--episode-hours", str(EPISODE_HOURS),
            "--reward-scale", str(REWARD_SCALE),
            "--variant", VARIANT,
            "--device", DEVICE,
            "--window-indices", "0", "1", "2",
            "--scenario-protocol", SCENARIO_PROTOCOL,
            "--hard-scenario-probability", "0.5",
            "--forecast-context-hours", "168",
            "--future-summary-windows-h", "24", "72",
            "--stress-level", "medium",
        ]
        policy_data.generate_dataset(policy_data.parse_args(argv))

    generate_g1(G1_TRAIN, "train", [1600, 1601])
    generate_g1(G1_VALIDATION, "validation", [3300])

    if not (P2_DIR / "iterative_action_q.pt").exists():
        p2_args = trainer.parse_args(
            [
                "--train-data", str(TRAIN_DATA), str(G1_TRAIN),
                "--validation-data", str(VALIDATION_DATA), str(G1_VALIDATION),
                "--initial-checkpoint", str(CHECKPOINT),
                "--out-dir", str(P2_DIR),
                "--observation-input", "state_only",
                "--epochs", "12",
                "--patience", "4",
                "--batch-size", "4",
                "--heads", "5",
                "--quantiles", "21",
                "--model-seed", str(MODEL_SEED),
                "--device", DEVICE,
            ]
        )
        p2_summary = trainer.run(p2_args)
    else:
        p2_summary = json.loads(
            (P2_DIR / "summary.json").read_text(encoding="utf-8")
        )
    print(json.dumps(p2_summary["final_validation"], indent=2))
else:
    print("P2 skipped. Set RUN_P2=True to run policy roll-in data collection.")


## 13. Package and download the artifacts

A Colab runtime is temporary. This cell bundles datasets, checkpoints, summaries, evaluation tables, the hourly replay trace, and the standalone V2 cinematic dashboard. Download the archive or copy it to Google Drive.


In [ ]:
import shutil

archive_base = Path("/content/ccs_rl_iterative_q_demo") if IN_COLAB else RUN_ROOT
archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=RUN_ROOT,
    )
)
print(f"Artifact bundle: {archive_path} ({archive_path.stat().st_size / 1e6:.1f} MB)")

if IN_COLAB:
    from google.colab import files
    # Uncomment the next line to download the archive.
    # files.download(str(archive_path))


## 14. Scale the Colab workflow to the formal experiment

The notebook intentionally uses a small simulator budget. The production configuration differs as follows:

| Item | Colab quick demo | Formal Iterative Action-Q |
|---|---:|---:|
| Episode | 168 h | 720 h |
| G0 training seeds | 4 | 200 |
| G0 roots per seed | 3 | 8 |
| G1/G2/G3 policy seeds | optional 2 | 40 / 60 / 100 |
| Cumulative roots | small | 3,200 nominal roots |
| Quantiles / heads | 21 / 5 | 51 / 5 |
| Max epochs / patience | 15 / 4 | 40 / 8 |
| Model stages | P1, optional P2 | P1 → P2 → P3 → P4 |
| Evaluation | 3 demo seeds | 3 model seeds × 30 paired scenarios |

The full dependency graph is managed by `hpc/launch_iterative_action_q.sh`; the method specification is in `docs/iterative_action_q_training_zh.md`. A formal reproduction must follow the seed manifest, protocol locks, and artifact provenance. Never mix the notebook's demonstration seeds with the E1 formal-test block.

### Completed end-to-end path

```text
physical environment
  → legal joint-action masks
  → Greedy roll-in
  → same-state counterfactual full-episode labels
  → leakage-safe train/validation datasets
  → ensemble quantile Action-Q training
  → early-stopped checkpoint
  → confidence/margin gated deployment
  → paired held-out scenario evaluation
  → V2 cinematic operational replay
```
